# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
# load pdf
from langchain_community.document_loaders import PyPDFLoader

file_path = "documents/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

# Join all pages into a single document text
document_text = ""
for pg in docs:
    document_text += pg.page_content + "\n"

print(f"Document length: {len(document_text)} characters")

26
Document length: 53851 characters


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from pydantic import BaseModel
from openai import OpenAI
import os

In [4]:
# Output class for final result 
class Output(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

# Intermediate data
class DocumentAnalysis(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str

In [ ]:
client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

chosen_tone = "Formal Academic Writing"

# Store instructions and context 
system_prompt = f"""You are a specialist at summarizing documents. Your task is to analyze a document and extract key information, then provide a summary written in {chosen_tone} style. 

The summary should be:
- Concise and succinct (no longer than 1000 tokens)
- Written in Formal Academic Writing style (characterized by precise vocabulary, complex sentence structures, objective tone, and adherence to academic conventions)
- Professional and suitable for an AI professional audience

You must extract:
1. The author of the document
2. The title of the document
3. A relevance statement explaining why this article is relevant for AI professionals in their professional development (no longer than one paragraph)
4. A summary written in the specified tone"""

# Add document context via string formatting
user_prompt = f"""Please analyze the following document and provide the requested information:

Document:
{document_text}

Based on this document, please provide:
1. Author: The author(s) of this document
2. Title: The title of this document
3. Relevance: A statement (no longer than one paragraph) explaining why this article is relevant for an AI professional in their professional development
4. Summary: A concise and succinct summary (no longer than 1000 tokens) written in Formal Academic Writing style
5. Tone: The tone used for the summary ({chosen_tone})"""

# Parse output using the response API and DocumentAnalysis model (tokens from response)
response = client.responses.parse(
    model="gpt-4o-mini",
    instructions=system_prompt,
    input=user_prompt,
    text_format=DocumentAnalysis,
    max_output_tokens=2000
)

# Get structured data from response
parsed_output = response.output_parsed

# Build final output including token counts
output = Output(
    Author=parsed_output.Author,
    Title=parsed_output.Title,
    Relevance=parsed_output.Relevance,
    Summary=parsed_output.Summary,
    Tone=parsed_output.Tone,
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens
)


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [6]:
from deepeval.metrics import GEval, SummarizationMetric
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel


In [7]:
# instantiate a model
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

assessment_questions_summary = [
        "Does the summary capture the key findings about the state of AI adoption in business?",
        "Are the main statistics and data points from the report accurately represented?",
        "Does the summary include the main challenges, solutions, and recommendations discussed in the original document?",
        "Does the summary avoid introducing any information, claims, or details that are not present in the original document?",
        "Are important concepts and domain-specific information from the original document preserved in the summary?",
    ] 

summary_metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions= assessment_questions_summary
)

# Prepare evaluation test case
test_case = LLMTestCase(
    input=document_text,
    actual_output=output.Summary
)

# Run evaluation using measure() method to set score and reason
summary_metric.measure(test_case)

# Get score and explanation
SummarizationScore = summary_metric.score
SummarizationReason = summary_metric.reason

print(f"SummarizationScore: {SummarizationScore}")
print(f"SummarizationReason: {SummarizationReason}")

Output()

SummarizationScore: 0.55
SummarizationReason: The score is 0.55 because the summary contains significant contradictions to the original text regarding the capabilities of GenAI systems, which undermines its accuracy. Additionally, it introduces numerous points not found in the original text, leading to a misrepresentation of the core message.


In [8]:
# Configure GEval metrics to use the same model

# Coherence metric
coherence_evals = [
        "Evaluate whether the summary uses clear and direct language throughout.",
        "Check if complex ideas are presented in a logical sequence that's easy to follow.",
        "Assess whether each paragraph or section flows naturally into the next.",
        "Identify any vague, ambiguous, or confusing parts that reduce understanding.",
        "Determine if the summary maintains consistent terminology and avoids contradictory statements."
    ]
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=coherence_evals,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

# Tonality metric
tonality_evals = [
    "Determine whether the summary maintains the specified Formal Academic Writing tone throughout.",
    "Evaluate if the language reflects the formality and expertise appropriate for academic writing.",
    "Check if the vocabulary and sentence structure match Formal Academic Writing conventions.",
    "Assess whether the summary avoids casual expressions, slang, or overly informal phrasing.",
    "Verify that the tone is consistent across all sections and maintains professional objectivity."
]
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=tonality_evals,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

# Safety metric
safety_evals = [
    "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
    "Identify any content that could be harmful, discriminatory, or promote unsafe practices.",
    "Assess whether the summary contains any biased statements or stereotypes.",
    "Evaluate if the language is respectful and avoids offensive or inappropriate content.",
    "Verify that the summary does not contain any misleading information that could cause harm."
]
safety_metric = GEval(
    name="Safety",
    evaluation_steps=safety_evals,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

# Test case with summary as actual output
test_case = LLMTestCase(
    input=document_text,
    actual_output=output.Summary
)

# Evaluate all GEval metrics using measure() method to set score and reason
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

# Get scores and reasons
CoherenceScore = coherence_metric.score
CoherenceReason = coherence_metric.reason

TonalityScore = tonality_metric.score
TonalityReason = tonality_metric.reason

SafetyScore = safety_metric.score
SafetyReason = safety_metric.reason

print(f"\nCoherenceScore: {CoherenceScore}")
print(f"CoherenceReason: {CoherenceReason}\n")

print(f"TonalityScore: {TonalityScore}")
print(f"TonalityReason: {TonalityReason}\n")

print(f"SafetyScore: {SafetyScore}")
print(f"SafetyReason: {SafetyReason}\n")

# Structured output dictionary
evaluation_results = {
    "SummarizationScore": SummarizationScore,
    "SummarizationReason": SummarizationReason,
    "CoherenceScore": CoherenceScore,
    "CoherenceReason": CoherenceReason,
    "TonalityScore": TonalityScore,
    "TonalityReason": TonalityReason,
    "SafetyScore": SafetyScore,
    "SafetyReason": SafetyReason
}

print("\nStructured Evaluation Results:")
for key, value in evaluation_results.items():
    print(f"{key}: {value}")

Output()

Output()

Output()


CoherenceScore: 0.8562176500885798
CoherenceReason: The summary uses clear and direct language, effectively presenting complex ideas in a logical sequence. Each section flows naturally into the next, maintaining coherence throughout. While the content is rich and informative, there are minor areas where terminology could be more consistent, particularly regarding the distinction between front-office and back-office applications. Overall, the summary is well-structured and comprehensible, with only slight ambiguity in some terms.

TonalityScore: 0.9047924390851522
TonalityReason: The summary maintains a formal academic writing tone throughout, using precise language and complex sentence structures that reflect expertise. It avoids casual expressions and slang, adhering to the conventions of academic writing. The vocabulary is appropriate for the subject matter, and the tone remains consistent, demonstrating professional objectivity. However, a slight improvement could be made in furthe

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
# System prompt for summary improvement
enhancement_system_prompt = """You are an expert at improving document summaries based on evaluation feedback. 
Your task is to analyze the original summary, review the evaluation feedback, and produce an enhanced version that addresses the identified weaknesses.
"""

assessment_questions_summary_text = ''
for item in assessment_questions_summary:
    assessment_questions_summary_text += item + '\n'

coherence_evals_text = ''
for item in coherence_evals:
    coherence_evals_text += item + '\n'

tonality_evals_text = ''
for item in tonality_evals:
    tonality_evals_text += item + '\n'

safety_evals_text = ''
for item in safety_evals:
    safety_evals_text += item + '\n'


# Prepare user prompt with document, summary, and evaluation
enhancement_user_prompt = f"""Please improve the following summary based on the prompt used for generate the original summary and the evaluation feedback provided.

Original System Prompt:
{system_prompt}

Original User Prompt:
{user_prompt}

Original Document Context:
{document_text}

Original Summary:
{output.Summary}

Evaluation criteria:
- Summarization: {assessment_questions_summary_text}
- Coherence: {coherence_evals_text}
- Tonality: {tonality_evals_text}
- Safety: {safety_evals_text}

Evaluation Feedback:
- Summarization Score: {SummarizationScore:.3f}
  Reason: {SummarizationReason}

- Coherence Score: {CoherenceScore:.3f}
  Reason: {CoherenceReason}

- Tonality Score: {TonalityScore:.3f}
  Reason: {TonalityReason}

- Safety Score: {SafetyScore:.3f}
  Reason: {SafetyReason}


The enhanced summary must:
- Maintain the same Formal Academic Writing tone as the original
- Address specific issues mentioned in the evaluation feedback
- Improve areas where scores were lower
- Preserve all important information from the original document
"""

# Generate the enhanced summary using the same model for consistency
enhanced_response = client.responses.parse(
    model="gpt-4o-mini",
    instructions=enhancement_system_prompt,
    input=enhancement_user_prompt,
    text_format=DocumentAnalysis,
    max_output_tokens=1500  
)

# Get the improved summary
enhanced_summary_text = enhanced_response.output_parsed.Summary

# Re-evaluate the enhanced summary 

# Create test case for enhanced summary
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary_text
)

# Reuse the original summary_metric to evaluate the enhanced summary
summary_metric.measure(enhanced_test_case)
EnhancedSummarizationScore = summary_metric.score
EnhancedSummarizationReason = summary_metric.reason

# Reuse the original GEval metrics to evaluate the enhanced summary
coherence_metric.measure(enhanced_test_case)
EnhancedCoherenceScore = coherence_metric.score
EnhancedCoherenceReason = coherence_metric.reason

tonality_metric.measure(enhanced_test_case)
EnhancedTonalityScore = tonality_metric.score
EnhancedTonalityReason = tonality_metric.reason

safety_metric.measure(enhanced_test_case)
EnhancedSafetyScore = safety_metric.score
EnhancedSafetyReason = safety_metric.reason

# Organize enhanced evaluation results
enhanced_evaluation_results = {
    "SummarizationScore": EnhancedSummarizationScore,
    "SummarizationReason": EnhancedSummarizationReason,
    "CoherenceScore": EnhancedCoherenceScore,
    "CoherenceReason": EnhancedCoherenceReason,
    "TonalityScore": EnhancedTonalityScore,
    "TonalityReason": EnhancedTonalityReason,
    "SafetyScore": EnhancedSafetyScore,
    "SafetyReason": EnhancedSafetyReason
}

Output()

Output()

Output()

Output()

In [ ]:
# Compare original and enhanced scores
print("COMPARISON: ORIGINAL vs ENHANCED SUMMARY")

comparison_results = {}
for key in ["SummarizationScore", "CoherenceScore", "TonalityScore", "SafetyScore"]:
    original_score = evaluation_results[key]
    enhanced_score = enhanced_evaluation_results[key]
    original_reason = evaluation_results[key.replace('Score', 'Reason')]
    enhanced_reason = enhanced_evaluation_results[key.replace('Score', 'Reason')]
    improvement = enhanced_score - original_score
    comparison_results[key] = {
        "Original": original_score,
        "Enhanced": enhanced_score,
        "Improvement": improvement,
        "Original_reasoning": original_reason,
        "Enhanced_reasoning": enhanced_reason
    }
    print(f"\n{key}:")
    print(f"Original:  {original_score:.3f}")
    print(f"Enhanced:  {enhanced_score:.3f}")
    print(f"Change:    {improvement:+.3f}")
    print(f"Original reason: {original_reason}")
    print(f"Current reason: {enhanced_reason}")

print("""
The SummarizationScore improved but all other three metrics dropped slightly. The increase in summarization score seems to result from
a reduction of inconsistent information from the summary, which may scarifice the other aspects as the model is trying to achieve a balance 
among all metrics at the same time. Improving the text for one metric every time may better increase the scores. 

The current controls may not be enough as the total score is still relatively low and the evaluation reason for the summarizationscore states that 
there are inconsistent and new information introduced in the summary. Also it is not certain whether this pipeline can reliably work for different texts
or not.
""")


COMPARISON: ORIGINAL vs ENHANCED SUMMARY

SummarizationScore:
Original:  0.550
Enhanced:  0.667
Change:    +0.117
Original reason: The score is 0.55 because the summary contains significant contradictions to the original text regarding the capabilities of GenAI systems, which undermines its accuracy. Additionally, it introduces numerous points not found in the original text, leading to a misrepresentation of the core message.
Current reason: The score is 0.67 because the summary contains contradictions regarding the investment amounts in GenAI and introduces extra information that was not present in the original text, which affects its accuracy and completeness.

CoherenceScore:
Original:  0.856
Enhanced:  0.816
Change:    -0.040
Original reason: The summary uses clear and direct language, effectively presenting complex ideas in a logical sequence. Each section flows naturally into the next, maintaining coherence throughout. While the content is rich and informative, there are minor ar

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
